## B9 · GroupBy, aggregate and apply

Splitting data into groups, summarising each group, and applying functions.

In [749]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Plot defaults: wide 16:9 figures, sharp ("retina") output and slightly smaller text
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({
    "figure.figsize": (6.4, 3.6),
    "figure.dpi": 100,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

# Same data as sns.load_dataset('penguins'), saved locally so it works offline
df_penguins = pd.read_csv("data/penguins.csv")

# Complete rows only (no missing values) so that group statistics are readable
df_complete = df_penguins.dropna().copy()

### Summarising by group

#### Grouping and summarising with `groupby`

**B9.01**

`df.groupby('group')['col'].mean()`

***(non-examinable)***

The `groupby()` method allows us to split rows into groups, calculate a statistic for each group and combine the results in one table. It is useful for comparing categories in one line. Alternatively, build a mask for each group (see the exploring sheet) when only one or two groups are needed.

In [750]:
# display the example data
df_complete[["species", "body_mass_g"]].head()

,species,body_mass_g
0,Adelie,3750.0
1,Adelie,3800.0
2,Adelie,3250.0
4,Adelie,3450.0
5,Adelie,3650.0


In [751]:
df_complete.groupby("species")["body_mass_g"].mean()

species
Adelie       3706.164384
Chinstrap    3733.088235
Gentoo       5092.436975
Name: body_mass_g, dtype: float64

*See also:* Comparing groups with masks (B5) · Calculating several statistics per group (B9) · Counting rows per group (B9) · Grouping by several columns (B9)

#### Grouping by several columns

**B9.02**

`df.groupby(['group_1', 'group_2'])['col'].mean()`

***(non-examinable)***

Passing a list to `groupby` allows us to split by a combination of columns, such as species *and* sex. It is useful when two factors both matter. Alternatively, `pd.pivot_table` lays the same numbers out as a grid.

In [752]:
# display the example data
df_complete[["species", "sex", "body_mass_g"]].head()

,species,sex,body_mass_g
0,Adelie,Male,3750.0
1,Adelie,Female,3800.0
2,Adelie,Female,3250.0
4,Adelie,Female,3450.0
5,Adelie,Male,3650.0


In [753]:
df_complete.groupby(["species", "sex"])["body_mass_g"].mean().round(0)

species    sex   
Adelie     Female    3369.0
           Male      4043.0
Chinstrap  Female    3527.0
           Male      3939.0
Gentoo     Female    4680.0
           Male      5485.0
Name: body_mass_g, dtype: float64

*See also:* Grouping and summarising with `groupby` (B9) · Summarising into a grid with `pivot_table` (B9) · Counting rows per group (B9)

#### Counting rows per group

**B9.03**

`g.size(), g['c'].count()`

***(non-examinable)***

The `.size()` and `.count()` methods allow us to count the rows in each group. They are useful for sample sizes. Alternatively, `value_counts()` counts a single column without `groupby`.

`size()` counts rows; `count()` counts non-missing values in a column, so the two differ when data are missing. This example uses the original `df_penguins` table, which has missing sexes.

In [754]:
g = df_penguins.groupby("species")
g.size()

species
Adelie       152
Chinstrap     68
Gentoo       124
dtype: int64

In [755]:
g['sex'].count()

species
Adelie       146
Chinstrap     68
Gentoo       119
Name: sex, dtype: int64

*See also:* Counting missing values (B5) · Grouping and summarising with `groupby` (B9)

#### Calculating several statistics per group

**B9.04**

`g['c'].agg(['mean', 'std', 'count'])`

***(non-examinable)***

The `.agg()` method allows us to calculate several statistics per group in one go. It is useful for building a summary table with mean, spread and count. Alternatively, call each statistic separately and combine with `pd.concat`.

In [756]:
# display the example data
df_complete[["species", "body_mass_g"]].head()

,species,body_mass_g
0,Adelie,3750.0
1,Adelie,3800.0
2,Adelie,3250.0
4,Adelie,3450.0
5,Adelie,3650.0


In [757]:
df_complete.groupby("species")["body_mass_g"].agg(["mean", "std", "count"]).round(1)

,mean,std,count
species,,,
Adelie,3706.2,458.6,146
Chinstrap,3733.1,384.3,68
Gentoo,5092.4,501.5,119


*See also:* Grouping and summarising with `groupby` (B9) · Using named aggregation (B9) · Building a summary table (B5)

#### Using named aggregation

**B9.05**

`agg(new_name=('col', 'mean'))`

***(non-examinable)***

Named aggregation allows us to calculate different statistics for different columns and name the result columns. It is useful for tidy, report-ready tables. Alternatively, pass a dictionary: `agg({"col": "mean"})`.

In [758]:
# display the example data
df_complete[["species", "body_mass_g", "flipper_length_mm", "sex"]].head()

,species,body_mass_g,flipper_length_mm,sex
0,Adelie,3750.0,181.0,Male
1,Adelie,3800.0,186.0,Female
2,Adelie,3250.0,195.0,Female
4,Adelie,3450.0,193.0,Female
5,Adelie,3650.0,190.0,Male


In [759]:
df_complete.groupby("species").agg(
    mean_mass=("body_mass_g", "mean"),
    max_flipper=("flipper_length_mm", "max"),
    n=("sex", "count"),
).round(1)

,mean_mass,max_flipper,n
species,,,
Adelie,3706.2,210.0,146
Chinstrap,3733.1,212.0,68
Gentoo,5092.4,231.0,119


*See also:* Calculating several statistics per group (B9) · Grouping and summarising with `groupby` (B9) · Returning a flat table from groupby (B9)

#### Returning a flat table from groupby

**B9.06**

`as_index=False, .reset_index()`

***(non-examinable)***

The `as_index=False` option allows us to keep the group names as an ordinary column rather than the index. It is useful for plotting with seaborn or merging with other tables. Alternatively, call `.reset_index()` on the result.

In [760]:
df_complete.groupby('species')['body_mass_g'].mean()       # species is the index

species
Adelie       3706.164384
Chinstrap    3733.088235
Gentoo       5092.436975
Name: body_mass_g, dtype: float64

In [761]:
df_complete.groupby("species", as_index=False)["body_mass_g"].mean()    # species is a column

,species,body_mass_g
0,Adelie,3706.164384
1,Chinstrap,3733.088235
2,Gentoo,5092.436975


*See also:* Grouping and summarising with `groupby` (B9) · Summarising by two groups (B9)

#### Summarising by two groups

**B9.07**

`df.groupby(['group_1', 'group_2'])['col'].agg(funcs)`

***(non-examinable)***

Grouping by two columns and aggregating allows us to reproduce the "species by sex" comparison in one line. It is useful for comparing every combination of two categories. Alternatively, `pivot_table` shows one statistic in a grid.

In [762]:
# display the example data
df_complete[["species", "sex", "body_mass_g"]].head()

,species,sex,body_mass_g
0,Adelie,Male,3750.0
1,Adelie,Female,3800.0
2,Adelie,Female,3250.0
4,Adelie,Female,3450.0
5,Adelie,Male,3650.0


In [763]:
df_complete.groupby(["species", "sex"])["body_mass_g"].agg(["mean", "std", "count"]).round(1)

mean    std  count
species   sex                         
Adelie    Female  3368.8  269.4     73
          Male    4043.5  346.8     73
Chinstrap Female  3527.2  285.3     34
          Male    3939.0  362.1     34
Gentoo    Female  4679.7  281.6     58
          Male    5484.8  313.2     61

*See also:* Summarising into a grid with `pivot_table` (B9) · Building a summary table (B5) · Comparing groups with masks (B5)

#### Summarising into a grid with `pivot_table`

**B9.08**

`pd.pivot_table(df, index='col_1', columns='col_2', values='col_3', aggfunc='mean')`

***(non-examinable)***

The `pd.pivot_table()` function allows us to summarise a value across two categories laid out as a grid. It is useful for cross-tab style reports, and, unlike `pivot`, it can combine repeated entries. Alternatively, `groupby(...).mean().unstack()` gives the same layout.

In [764]:
# display the example data
df_complete[["species", "sex", "body_mass_g"]].head()

,species,sex,body_mass_g
0,Adelie,Male,3750.0
1,Adelie,Female,3800.0
2,Adelie,Female,3250.0
4,Adelie,Female,3450.0
5,Adelie,Male,3650.0


In [765]:
pd.pivot_table(df_complete, index="species", columns="sex", values="body_mass_g", aggfunc="mean").round(0)

sex,Female,Male
species,,
Adelie,3369.0,4043.0
Chinstrap,3527.0,3939.0
Gentoo,4680.0,5485.0


*See also:* Grouping and summarising with `groupby` (B9) · Reshaping long to wide with `pivot` and `index` (B6) · Grouping by several columns (B9) · Counting two categories together (B5)

### Transform and loop

#### Adding a group statistic to every row

**B9.09**

`g['c'].transform('mean')`

***(non-examinable)***

The `.transform()` method allows us to calculate a group statistic and give it back with the *same length* as the original, so it can be added as a column. It is useful for deviations from a group mean. Alternatively, merge a `groupby` result back onto the table.

In [766]:
df_species_mass = df_complete[["species", "body_mass_g"]].copy()
df_species_mass.head()                        # before

,species,body_mass_g
0,Adelie,3750.0
1,Adelie,3800.0
2,Adelie,3250.0
4,Adelie,3450.0
5,Adelie,3650.0


In [767]:
df_species_mass["species_mean"] = df_species_mass.groupby("species")["body_mass_g"].transform("mean")
df_species_mass["difference"] = df_species_mass["body_mass_g"] - df_species_mass["species_mean"]
df_species_mass.head()                        # after: the group mean is added to every row

,species,body_mass_g,species_mean,difference
0,Adelie,3750.0,3706.164384,43.835616
1,Adelie,3800.0,3706.164384,93.835616
2,Adelie,3250.0,3706.164384,-456.164384
4,Adelie,3450.0,3706.164384,-256.164384
5,Adelie,3650.0,3706.164384,-56.164384


*See also:* Adding a column from two columns (B4) · Grouping and summarising with `groupby` (B9) · Looping over groups (B9)

#### Looping over groups

**B9.10**

`for name, df_group in df.groupby('group'):`

***(non-examinable)***

Looping over a `groupby` object allows us to handle each group as its own small DataFrame. It is useful for per-group plots or tests. Alternatively, `.apply(func)` runs a function on every group.

In [768]:
for name, df_group in df_complete.groupby("species"):
    print(f'{name} - {len(df_group)} penguins, mean mass {round(df_group["body_mass_g"].mean())}')

Adelie - 146 penguins, mean mass 3706
Chinstrap - 68 penguins, mean mass 3733
Gentoo - 119 penguins, mean mass 5092


*See also:* Looping over a dictionary (A3) · Applying a function (B9) · Adding a group statistic to every row (B9) · Grouping and summarising with `groupby` (B9)

### map and apply

#### Mapping values with a dictionary or function

**B9.11**

`s.map(dict_or_func)`

***(non-examinable)***

The `.map()` method allows us to replace each value using a dictionary or a function. It is useful for recoding categories. Alternatively, `.replace()` leaves unmatched values unchanged.

Adelie, Chinstrap and Gentoo are the three penguin species in this dataset.

In [769]:
# display the example data
df_complete["species"].head()

0    Adelie
1    Adelie
2    Adelie
4    Adelie
5    Adelie
Name: species, dtype: object

In [770]:
df_complete["species"].map({"Adelie": "A", "Gentoo": "G", "Chinstrap": "C"}).head()

0    A
1    A
2    A
4    A
5    A
Name: species, dtype: object

*See also:* Replacing and mapping values (B4) · Recoding values with a dictionary (A3) · Applying a function (B9)

#### Applying a function

**B9.12**

`s.apply(func), df.apply(func, axis=1)`

***(non-examinable)***

The `.apply()` method allows us to run any function on every value, or on every row with `axis=1`. It is useful when no built-in method does the job. Alternatively, vectorised operations (`df['c'] / 1000`) are shorter and much faster.

A `lambda` is a small throw-away function.

In [771]:
df_small = df_complete[["species", "island", "body_mass_g"]].head(3)
df_small

,species,island,body_mass_g
0,Adelie,Torgersen,3750.0
1,Adelie,Torgersen,3800.0
2,Adelie,Torgersen,3250.0


In [772]:
df_small['body_mass_g'].apply(lambda g: g / 1000).tolist()        # per value (but g / 1000 is faster)

[3.75, 3.8, 3.25]

In [773]:
df_small.apply(lambda row: row["species"] + " on " + row["island"], axis=1)   # per row

0    Adelie on Torgersen
1    Adelie on Torgersen
2    Adelie on Torgersen
dtype: object

*See also:* Writing a function with `def` (A2) · Adding a calculated column (B4) · Mapping values with a dictionary or function (B9) · Checking a new column against a hand calculation (B1)

# Part C – Visualisation